# Proyecto completo de ML: Modelo de aprobación de tarjetas de crédito

## 0. Configuración
Importamos las librerías básicas que vamos a usar y dejamos guardadas las variables fijas del proyecto, como la semilla aleatoria y los nombres de los archivos del dataset.

In [1]:
import os
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import set_config

# Ponemos la vista en modo diagrama para ver los pipelines como cajas
set_config(display="diagram")

# Guardamos aqui los valores fijos que vamos a usar en todo el cuaderno
SEMILLA = 42
PORCENTAJE_TEST = 0.20
NOMBRE_DATASET = "rohitudageri/credit-card-details"
ARCHIVO_CLIENTES = "Credit_card.csv"
ARCHIVO_ETIQUETAS = "Credit_card_label.csv"

/home/carlos/Sistemas_Aprendizaje_Automatico/02_tarjetas_credito/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carga y unión de los datos

### 1.1 Descarga y carga de los CSV
Descargamos el dataset *Credit Card Details* desde Kaggle y cargamos por separado `Credit_card.csv` (datos de los clientes) y `Credit_card_label.csv` (resultado de la solicitud: 0 = aprobada, 1 = rechazada) para ver su tamaño y sus columnas antes de unirlos.

In [2]:
# Bajamos los archivos del dataset desde Kaggle
ruta_datos = kagglehub.dataset_download(NOMBRE_DATASET)
print("Ruta de los archivos:", ruta_datos)
print("Archivos descargados:", os.listdir(ruta_datos))

# Leemos los dos archivos csv por separado
datos_clientes = pd.read_csv(os.path.join(ruta_datos, ARCHIVO_CLIENTES))
datos_etiquetas = pd.read_csv(os.path.join(ruta_datos, ARCHIVO_ETIQUETAS))

# Miramos cuantas filas y columnas tiene cada tabla
print("\nForma de datos_clientes:", datos_clientes.shape)
print("Forma de datos_etiquetas:", datos_etiquetas.shape)

# Sacamos la lista con los nombres de todas las columnas
print("\nColumnas clientes:", list(datos_clientes.columns))
print("Columnas etiquetas:", list(datos_etiquetas.columns))

# Mostramos las 3 primeras filas de cada tabla para ver como son
display(datos_clientes.head(3))
display(datos_etiquetas.head(3))

100%|██████████| 33.5k/33.5k [00:00<00:00, 602kB/s]

Extracting files...
Ruta de los archivos: /home/carlos/.cache/kagglehub/datasets/rohitudageri/credit-card-details/versions/1
Archivos descargados: ['Credit_card.csv', 'Credit_card_label.csv']

Forma de datos_clientes: (1548, 18)
Forma de datos_etiquetas: (1548, 2)

Columnas clientes: ['Ind_ID', 'GENDER', 'Car_Owner', 'Propert_Owner', 'CHILDREN', 'Annual_income', 'Type_Income', 'EDUCATION', 'Marital_status', 'Housing_type', 'Birthday_count', 'Employed_days', 'Mobile_phone', 'Work_Phone', 'Phone', 'EMAIL_ID', 'Type_Occupation', 'Family_Members']
Columnas etiquetas: ['Ind_ID', 'label']


,Ind_ID,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Birthday_count,Employed_days,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members
0,5008827,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,-18772.0,365243,1,0,0,0,NaN,2
1,5009744,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2
2,5009746,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,NaN,-586,1,1,1,0,NaN,2


,Ind_ID,label
0,5008827,1
1,5009744,1
2,5009746,1


**Conclusiones:**
* Ambos ficheros tienen exactamente **1.548 filas**: `Credit_card.csv` contiene 18 columnas con información del solicitante y `Credit_card_label.csv` contiene 2 columnas (`Ind_ID` y `label`).
* Las dos tablas comparten la columna `Ind_ID`, que actuará como clave de unión.
* En esta primera vista ya se detectan detalles que habrá que revisar más adelante: valores nulos (`NaN` en `Type_Occupation` y `Birthday_count`), días en formato negativo (`Birthday_count` y `Employed_days`) y un valor anómalo positivo (`365243`) en `Employed_days` para un cliente pensionista.

### 1.2 Comprobación de la clave de unión y duplicados
Antes de unir las dos tablas, comprobamos si `Ind_ID` tiene valores nulos, si es único en cada archivo y si todos los identificadores coinciden entre ambos CSV. También revisamos si hay filas duplicadas exactas o clientes con los mismos datos registrados bajo distinto `Ind_ID`.

In [3]:
# Miramos si hay huecos vacios en la columna Ind_ID de cada tabla
print("Nulos en Ind_ID clientes:", datos_clientes["Ind_ID"].isna().sum())
print("Nulos en Ind_ID etiquetas:", datos_etiquetas["Ind_ID"].isna().sum())

# Comprobamos cuantos codigos distintos hay en cada tabla
print("\nIDs unicos en clientes:", datos_clientes["Ind_ID"].nunique())
print("IDs unicos en etiquetas:", datos_etiquetas["Ind_ID"].nunique())

# Convertimos los IDs a conjuntos para ver si coinciden todos entre si
ids_clientes = set(datos_clientes["Ind_ID"])
ids_etiquetas = set(datos_etiquetas["Ind_ID"])

solo_en_clientes = ids_clientes - ids_etiquetas
solo_en_etiquetas = ids_etiquetas - ids_clientes

print("\nIDs que estan solo en clientes:", len(solo_en_clientes))
print("IDs que estan solo en etiquetas:", len(solo_en_etiquetas))
print("IDs que coinciden en las dos tablas:", len(ids_clientes & ids_etiquetas))

# Revisamos si hay filas repetidas exactamente iguales
print("\nFilas duplicadas exactas en clientes:", datos_clientes.duplicated().sum())
print("Filas duplicadas exactas en etiquetas:", datos_etiquetas.duplicated().sum())

# Miramos tambien si hay clientes repetidos con los mismos datos pero distinto Ind_ID
columnas_sin_id = [columna for columna in datos_clientes.columns if columna != "Ind_ID"]
duplicados_sin_id = datos_clientes.duplicated(subset=columnas_sin_id).sum()
print("Filas repetidas en clientes sin contar Ind_ID:", duplicados_sin_id)

Nulos en Ind_ID clientes: 0
Nulos en Ind_ID etiquetas: 0

IDs unicos en clientes: 1548
IDs unicos en etiquetas: 1548

IDs que estan solo en clientes: 0
IDs que estan solo en etiquetas: 0
IDs que coinciden en las dos tablas: 1548

Filas duplicadas exactas en clientes: 0
Filas duplicadas exactas en etiquetas: 0
Filas repetidas en clientes sin contar Ind_ID: 171


**Conclusiones:**
* La clave `Ind_ID` es válida y cumple una relación exacta **1 a 1**: no tiene nulos, los 1.548 valores son únicos en cada tabla y todos coinciden entre ambos archivos (0 registros sueltos).
* No hay filas duplicadas exactas en ninguno de los dos ficheros.
* Sin embargo, al ignorar `Ind_ID` aparecen **171 filas repetidas** en `datos_clientes`. Una vez unidas las tablas habrá que revisar si esos perfiles repetidos tienen la misma etiqueta (`label`) o etiquetas contradictorias para decidir qué hacer con ellos.

### 1.3 Unión de las tablas y análisis de perfiles repetidos
Para entrenar un modelo supervisado necesitamos que cada fila tenga tanto los datos del cliente como su resultado (`label`), así que unimos `datos_clientes` y `datos_etiquetas` mediante un **`inner` join** sobre `Ind_ID`. Como ya comprobamos en el paso anterior que los 1.548 códigos coinciden al 100 % en ambas tablas, sabemos con seguridad que este `inner` no descarta ningún registro.

En un entorno real de banco no haríamos un `inner` directamente a ciegas, sino que primero usaríamos un **`outer` join** para auditar posibles descuadres y no perder información: los clientes sin `label` podrían ser solicitudes todavía pendientes de respuesta (útiles luego para predecir) y las etiquetas sin cliente señalarían un fallo en la base de datos. Una vez apartados esos casos, al entrenamiento entrarían igualmente solo las solicitudes ya resueltas, que es el punto en el que ya estamos aquí tras la comprobación de 1.2.

Nada más unir las tablas, aprovechamos para revisar qué pasa con las **171 filas con perfiles repetidos** que vimos antes: comprobamos si tienen la misma etiqueta o si hay perfiles calcados con decisiones contradictorias (uno aprobado y otro rechazado).

In [4]:
# Unimos las dos tablas por Ind_ID usando inner y asegurando relacion 1 a 1
datos_completos = pd.merge(
    datos_clientes,
    datos_etiquetas,
    on="Ind_ID",
    how="inner",
    validate="one_to_one"
)

print("Forma de datos_completos:", datos_completos.shape)
display(datos_completos.head(3))

# Revisamos que pasa con las 171 filas repetidas ahora que ya tienen su label
columnas_cliente = [columna for columna in datos_clientes.columns if columna != "Ind_ID"]

# Contamos cuantas filas son iguales tanto en datos del cliente como en la etiqueta
repetidos_misma_etiqueta = datos_completos.duplicated(subset=columnas_cliente + ["label"]).sum()

# Agrupamos por perfil de cliente (sin quitar filas con nulos) para ver si alguno tiene 0 y 1 a la vez
etiquetas_por_perfil = datos_completos.groupby(columnas_cliente, dropna=False)["label"].nunique()
perfiles_contradictorios = (etiquetas_por_perfil > 1).sum()

# Sacamos cuantas filas en total pertenecen a esos perfiles con etiquetas distintas
filas_contradictorias = datos_completos.groupby(columnas_cliente, dropna=False)["label"].transform("nunique") > 1

print("\nFilas repetidas con los mismos datos y la misma etiqueta:", repetidos_misma_etiqueta)
print("Perfiles con los mismos datos pero distinta etiqueta (0 y 1):", perfiles_contradictorios)
print("Total de filas afectadas por esa contradiccion:", filas_contradictorias.sum())

# Si hay contradicciones, mostramos un ejemplo para verlo bien
if perfiles_contradictorios > 0:
    print("\nEjemplo de filas con los mismos datos pero etiqueta distinta:")
    display(datos_completos[filas_contradictorias].sort_values(by=columnas_cliente).head(6))

Forma de datos_completos: (1548, 19)


,Ind_ID,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Birthday_count,Employed_days,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label
0,5008827,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,-18772.0,365243,1,0,0,0,NaN,2,1
1,5009744,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2,1
2,5009746,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,NaN,-586,1,1,1,0,NaN,2,1



Filas repetidas con los mismos datos y la misma etiqueta: 162
Perfiles con los mismos datos pero distinta etiqueta (0 y 1): 9
Total de filas afectadas por esa contradiccion: 22

Ejemplo de filas con los mismos datos pero etiqueta distinta:


,Ind_ID,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Birthday_count,Employed_days,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label
147,5135485,F,N,Y,0,135000.0,State servant,Higher education,Single / not married,Municipal apartment,-10660.0,-1935,1,0,0,0,Core staff,1,1
148,5135487,F,N,Y,0,135000.0,State servant,Higher education,Single / not married,Municipal apartment,-10660.0,-1935,1,0,0,0,Core staff,1,1
459,5135484,F,N,Y,0,135000.0,State servant,Higher education,Single / not married,Municipal apartment,-10660.0,-1935,1,0,0,0,Core staff,1,0
166,5146444,F,N,Y,0,135000.0,Working,Secondary / secondary special,Widow,House / apartment,-14394.0,-919,1,0,1,0,Managers,1,1
899,5146447,F,N,Y,0,135000.0,Working,Secondary / secondary special,Widow,House / apartment,-14394.0,-919,1,0,1,0,Managers,1,0
138,5125708,F,Y,Y,0,135000.0,Working,Lower secondary,Married,House / apartment,-16413.0,-622,1,0,1,0,High skill tech staff,2,1


**Conclusiones:**
* La tabla unida `datos_completos` tiene **1.548 filas y 19 columnas**, manteniendo todos los registros originales sin pérdida de datos.
* Al analizar las 171 filas con perfiles repetidos junto con su `label`, vemos dos casos distintos:
  * **162 filas** tienen exactamente los mismos datos de cliente y también la misma etiqueta.
  * **9 perfiles** (que afectan a **22 filas** en total) tienen exactamente los mismos datos de cliente pero etiquetas contradictorias (`0` y `1` a la vez), con `Ind_ID` muy cercanos entre sí (por ejemplo `5135484`, `5135485` y `5135487`).
* Queda pendiente decidir si se mantiene este `inner` o se cambia a un esquema con `outer` de auditoría, y cómo tratar tanto los perfiles repetidos con misma etiqueta como los 9 perfiles contradictorios antes de pasar al análisis de nulos.